# 04 Data Joining — Automated（整合版）

**Mirrors the CONFIG/engine structure of `01`/`02`/`03_data_wrangling_auto.ipynb`.**

**目的**（跟 R 版本 `04_data_Joining.qmd` 的 Executive Summary 一致）：把 03 清理好的三個資料集合併成一份分析用的資料表。D1（Billboard 排行榜）是主表，D2（Spotify 音檔特徵）是主要合併（primary join），D3（歌詞主題/情緒資料）是輔助合併（auxiliary join）。

- **CONFIG（要改的東西）**：`JOIN_STAGES`——每一階段合併要帶哪些欄位過來、用哪一欄當「有沒有配對成功」的判斷依據
- **Engine（不用改）**：`dedup_and_left_join()`，兩次合併共用同一段邏輯（去重 + left join + 配對率）
- **輸出**：D1+D2 合併 → 配對率評估 → 未配對樣本檢查 → D1+D2+D3 合併 → 完整驗證 → 存檔

## 整體架構：三塊怎麼互相呼叫（先看大局，再看 Step 細節）

04 跟 01-03 不一樣的地方：01-03 是「同一段邏輯套用在 3 個資料集」，04 是「2 個不對稱的合併動作」（D1 是主表，D2、D3 分別合併一次，不是迴圈跑 3 次同樣的事）。**架構圖照實際結構畫，不硬套 01-03 的形狀**：

| ① CONFIG | | ② Engine（1 個通用 function） | | ③ Step 1-6（呼叫②，印報表+存檔） |
|---|:---:|---|:---:|---|
| `JOIN_STAGES`：兩階段合併分別要帶哪些欄位、用哪欄判斷配對成功 | → | `dedup_and_left_join()`<br>去重右表（依 `join_key`）+ left join + 印配對率，兩階段共用同一段邏輯 | → | Step 1：讀取 03 的輸出<br>Step 2：D1+D2 主要合併<br>Step 3：配對率評估（整體+依年代）<br>Step 4：未配對樣本檢查<br>Step 5：D1+D2+D3 輔助合併+完整驗證<br>Step 6：存檔 |

**這裡沒有「誰下結論」欄位**——跟 03 一樣的道理：04 不做新的清理判斷，只是執行合併+回報數字，該判斷的事（CLEANING_RULES 該清什麼）02 已經做完了。

## 04 在幹嘛？—— 6 步驟藍圖

| Step | 做什麼（對應 function） | 問的問題 |
|------|------|------|
| **Step 1** | 讀取 03 存的 `wrangled_D1/D2/D3.pkl` | 03 交出來的資料長什麼樣子？ |
| **Step 2** | D1+D2 主要合併（`dedup_and_left_join`） | D2 的音檔特徵（danceability、energy...）+ target + decade，接到 D1 每一筆排行榜紀錄上 |
| **Step 3** | 配對率評估 | D1 裡有多少比例真的在 D2 找到對應？依年代拆開看有沒有落差？ |
| **Step 4** | 未配對樣本檢查 | 配不到的那些長什麼樣子？是資料問題還是本來就不重疊？ |
| **Step 5** | D1+D2+D3 輔助合併+驗證 | 再把 D3 的 genre/歌詞主題資料接上去，配對率是多少？跟 D2 都配對成功的又有多少？ |
| **Step 6** | 存檔 | 存成最終分析用資料表，給 05 EDA / 06 建模接手 |

> **範圍提醒**：完全比照 R 版本的合併邏輯（D1 為主表、D2 主要合併、D3 輔助合併），沒有新增或減少合併範圍，只是把「去重+left join+配對率」這段共用邏輯抽成一個函數，不用像 R 一樣兩段分開手寫。

In [1]:
import pandas as pd

# Load wrangled datasets from Stage 03
d1 = pd.read_pickle('../Data/wrangled_D1.pkl')
d2 = pd.read_pickle('../Data/wrangled_D2.pkl')
d3 = pd.read_pickle('../Data/wrangled_D3.pkl')

print('[STEP 1] Load Wrangled Data（讀取 03 的輸出）')
print(f'D1 (Billboard): {len(d1):,} rows, {d1["join_key"].nunique():,} unique join_key')
print(f'D2 (Spotify):   {len(d2):,} rows, {d2["join_key"].nunique():,} unique join_key')
print(f'D3 (Music):     {len(d3):,} rows, {d3["join_key"].nunique():,} unique join_key')

[STEP 1] Load Wrangled Data（讀取 03 的輸出）
D1 (Billboard): 330,087 rows, 29,671 unique join_key
D2 (Spotify):   41,106 rows, 39,851 unique join_key
D3 (Music):     28,372 rows, 28,342 unique join_key


---
## ① CONFIG — `JOIN_STAGES`

**⚠️ 一個實戰批判性思考案例，不是憑空猜的**：D3 其實也有欄位叫 `danceability`、`loudness`、`acousticness`、`instrumentalness`、`valence`、`energy`（跟 D2 的 Spotify 音檔特徵撞名，但來源、算法完全不同）。R 版本在 Part 4 的 `select()` 裡**只挑了 12 個 D3 欄位（genre + 11 個歌詞主題欄位），刻意排除了這幾個撞名欄位**——查證後確認這不是疏漏，是刻意設計：如果兩邊撞名欄位都選進來，`left_join` 會被迫自動加 `_x`/`_y` 後綴，變成兩組意義不同但名字很像的音檔特徵混在一起，非常容易被誤用。**這裡照實移植 R 的選擇範圍，只帶 D3 的 12 個不撞名欄位過來。**

**另一個照實移植、沒有擅自修正的地方**：D2、D3 合併前都用 `distinct(join_key, .keep_all=TRUE)`（Python 對應 `drop_duplicates`）去重，保留的是**第一筆出現的紀錄，順序是任意的**（不是依某個規則選「最好的」那筆）。D2 有 41,106 筆但只有 39,794 個不重複 key，代表有約 1,300 筆重複 key 被任意保留其中一筆——這是 R 版本本來就有的限制，不是這次轉譯造成的，先照實移植，如果之後懷疑這個任意性影響到分析結果，再回頭處理。

In [2]:
# ============================================================
# CONFIG -- Only edit this section when changing what each join carries over
# ============================================================

JOIN_STAGES = {
    "D1_D2": {
        "right_df": d2,
        "right_name": "D2",
        # Spotify audio features + target label + decade (all from D2, no name collisions with D1)
        "carry_columns": [
            "danceability", "energy", "key", "loudness", "mode",
            "speechiness", "acousticness", "instrumentalness",
            "liveness", "valence", "tempo", "duration_ms",
            "time_signature", "chorus_hit", "sections",
            "target", "decade",
        ],
        "match_indicator_col": "target",  # non-null target = matched a D2 row
    },
    "D1D2_D3": {
        "right_df": d3,
        "right_name": "D3",
        # Only D3's 12 non-colliding columns -- deliberately excludes D3's own
        # danceability/energy/valence/etc, which share names with D2's audio
        # features but come from a different source. See CONFIG markdown above.
        "carry_columns": [
            "genre", "dating", "violence", "world/life", "night/time",
            "romantic", "communication", "obscene", "music",
            "sadness", "feelings", "topic",
        ],
        "match_indicator_col": "genre",  # non-null genre = matched a D3 row
    },
}

---
## ② Engine — No edits needed below this line

`dedup_and_left_join()` 是整份 04 唯一的合併邏輯，兩個合併階段（D1+D2、D1D2+D3）都呼叫同一個函數——它不知道自己在合併哪個資料集，只知道「給我一個右表 + 要帶哪些欄位 + 用哪個 key，我照著做並回報配對率」。

執行順序：
1. 右表依 `join_key` 去重（保留第一筆，跟 R 的 `distinct()` 行為一致）
2. 只選 `join_key` + 要帶過來的欄位，left join 到左表上
3. 用 `match_indicator_col` 判斷配對成功筆數，印出配對率

In [3]:
def dedup_and_left_join(left_df, right_df, right_name, carry_columns, match_indicator_col, join_key="join_key"):
    """Generic join step used by both D1+D2 and D1D2+D3 stages.
    Dedups the right-hand table on join_key (keeps first occurrence, matching
    R's distinct(join_key, .keep_all=TRUE) behaviour -- arbitrary tie-break,
    inherited from R, not something this function decides), left-joins the
    requested columns onto left_df, and reports the match rate."""

    before = len(right_df)
    right_dedup = right_df.drop_duplicates(subset=join_key, keep="first")
    print(f'  {right_name}: {before:,} rows -> {len(right_dedup):,} unique {join_key} (dropped {before - len(right_dedup):,} duplicates)')

    joined = left_df.merge(
        right_dedup[[join_key] + carry_columns],
        on=join_key,
        how="left",
    )

    total = len(joined)
    matched = joined[match_indicator_col].notna().sum()
    unmatched = total - matched
    print(f'  Total rows after join: {total:,}')
    print(f'  Matched:   {matched:,} rows ({round(matched / total * 100, 2)}%)')
    print(f'  Unmatched: {unmatched:,} rows ({round(unmatched / total * 100, 2)}%)')

    return joined

---
## ③ Step 2 — D1+D2 主要合併（對應藍圖表 Step 2）

In [4]:
print('[STEP 2] D1 + D2 Join (Primary Join)')

stage = JOIN_STAGES["D1_D2"]
d1_d2_joined = dedup_and_left_join(
    d1, stage["right_df"], stage["right_name"],
    stage["carry_columns"], stage["match_indicator_col"],
)

[STEP 2] D1 + D2 Join (Primary Join)
  D2: 41,106 rows -> 39,851 unique join_key (dropped 1,255 duplicates)


  Total rows after join: 330,087
  Matched:   264,478 rows (80.12%)
  Unmatched: 65,609 rows (19.88%)


---
## ③ Step 3 — 配對率評估：整體 + 依年代（對應藍圖表 Step 3）

**⚠️ 命名要小心，兩個「年代」不是同一件事**：合併進來的 `decade` 欄位是**從 D2（Spotify）帶過來的**（'60s'-'10s' 標籤）；這裡另外要算的 `decade_d1` 是**從 D1 自己的 `date` 欄位算出來的**（歌曲上榜的年代）。R 版本原文就是分開算、分開命名的，這裡延用同樣的命名避免搞混。

In [5]:
print('[STEP 3] D1+D2 Match Rate Evaluation（配對率評估）')

total = len(d1_d2_joined)
matched = d1_d2_joined["target"].notna().sum()
unmatched = total - matched
print(f'Total records: {total:,} rows')
print(f'Matched:       {matched:,} rows ({round(matched / total * 100, 2)}%)')
print(f'Unmatched:     {unmatched:,} rows ({round(unmatched / total * 100, 2)}%)')

# Match rate by D1's own chart-date decade (decade_d1), NOT the D2-sourced 'decade' column
d1_d2_joined["decade_d1"] = (d1_d2_joined["date"].dt.year // 10 * 10).astype(str) + "s"

decade_summary = (
    d1_d2_joined.groupby("decade_d1")
    .agg(total=("join_key", "size"), matched=("target", lambda s: s.notna().sum()))
    .reset_index()
)
decade_summary["match_pct"] = (decade_summary["matched"] / decade_summary["total"] * 100).round(2)
decade_summary = decade_summary.sort_values("decade_d1")
print()
print(decade_summary.to_string(index=False))

[STEP 3] D1+D2 Match Rate Evaluation（配對率評估）
Total records: 330,087 rows
Matched:       264,478 rows (80.12%)
Unmatched:     65,609 rows (19.88%)



decade_d1  total  matched  match_pct
    1950s   7400      489       6.61
    1960s  52100    36826      70.68
    1970s  52187    40933      78.44
    1980s  52200    44945      86.10
    1990s  52100    42576      81.72
    2000s  52200    48301      92.53
    2010s  52200    49156      94.17
    2020s   9700     1252      12.91


---
## ③ Step 4 — D1-D2 未配對樣本檢查（對應藍圖表 Step 4）

In [6]:
print('[STEP 4] D1-D2 Unmatched Records Sample Check（未配對樣本檢查）')

unmatched_sample = (
    d1_d2_joined[d1_d2_joined["target"].isna()]
    .assign(year=lambda df: df["date"].dt.year)
    .drop_duplicates(subset=["artist_clean", "song_clean", "year"])
    .sort_values("year")
    [["artist_clean", "song_clean", "year"]]
    .head(10)
)
print(unmatched_sample.to_string(index=False))

[STEP 4] D1-D2 Unmatched Records Sample Check（未配對樣本檢查）
                                        artist_clean                        song_clean  year
                                     frankie vaughan                              judy  1958
                                        the olympics  (i wanna) dance with the teacher  1958
                                       the four aces                 the world outside  1958
                                       jimmy clanton                      a part of me  1958
the tommy dorsey orchestra starring warren covington        i want to be happy cha cha  1958
                                           doris day                    tunnel of love  1958
                                         johnny cash                    all over again  1958
                               bernie lowe orchestra                 intermission riff  1958
                   johnny cash and the tennessee two i just thought you'd like to know  1958
               

---
## ③ Step 5 — D1+D2+D3 輔助合併 + 完整驗證（對應藍圖表 Step 5）

In [7]:
print('[STEP 5] D1 + D2 + D3 Join (Auxiliary Join)')

stage = JOIN_STAGES["D1D2_D3"]
d1_d2_d3_joined = dedup_and_left_join(
    d1_d2_joined, stage["right_df"], stage["right_name"],
    stage["carry_columns"], stage["match_indicator_col"],
)

print()
print('-- Full validation (matches R Part 4) --')
total = len(d1_d2_d3_joined)
d2_matched = d1_d2_d3_joined["target"].notna().sum()
d3_matched = d1_d2_d3_joined["genre"].notna().sum()
both_matched = (d1_d2_d3_joined["target"].notna() & d1_d2_d3_joined["genre"].notna()).sum()

print(f'Total rows:              {total:,}')
print(f'D2 matched (has target): {d2_matched:,} ({round(d2_matched / total * 100, 2)}%)')
print(f'D3 matched (has genre):  {d3_matched:,} ({round(d3_matched / total * 100, 2)}%)')
print(f'Both D2 + D3 matched:    {both_matched:,} ({round(both_matched / total * 100, 2)}%)')

print()
print('Note: D3 matched here (row-level, %) reads higher than 03 Step 5 D1<->D3 key-level',
      'overlap (10.63%) -- that number was computed on UNIQUE join_keys, this one counts every',
      'weekly chart row, so a song that matches D3 gets counted once per week it charted.')

[STEP 5] D1 + D2 + D3 Join (Auxiliary Join)
  D3: 28,372 rows -> 28,342 unique join_key (dropped 30 duplicates)


  Total rows after join: 330,087
  Matched:   46,028 rows (13.94%)
  Unmatched: 284,059 rows (86.06%)

-- Full validation (matches R Part 4) --
Total rows:              330,087
D2 matched (has target): 264,478 (80.12%)
D3 matched (has genre):  46,028 (13.94%)
Both D2 + D3 matched:    44,559 (13.5%)

Note: D3 matched here (row-level, %) reads higher than 03 Step 5 D1<->D3 key-level overlap (10.63%) -- that number was computed on UNIQUE join_keys, this one counts every weekly chart row, so a song that matches D3 gets counted once per week it charted.


---
## ④ Step 6 — 存檔（對應藍圖表 Step 6）

存成 `.pkl`，給 05 EDA / 06 建模接手。跟 01 Step 8、03 Step 6 一樣的邏輯。

In [8]:
print('[STEP 6] Save Final Analysis Dataset（存檔）')

save_path = r'..\Data\D1_D2_D3_joined.pkl'
d1_d2_d3_joined.to_pickle(save_path)

print(f'[OK] Saved to: {save_path}')
print(f'Total rows:    {len(d1_d2_d3_joined):,}')
print(f'Total columns: {d1_d2_d3_joined.shape[1]}')

[STEP 6] Save Final Analysis Dataset（存檔）


[OK] Saved to: ..\Data\D1_D2_D3_joined.pkl
Total rows:    330,087
Total columns: 40
